In [4]:
import os
os.chdir("../../web_backend/")

FileNotFoundError: [Errno 2] No such file or directory: '../../web_backend/'

In [5]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from data.data import CollectionAccessor, ImageHandler, EmbeddingSpaceAccessor

from search import Search, GraphSearcher, TextEmbeddingSearcher, EmbeddingSearcher

In [6]:
from app import init_DMG, search_collection

In [ ]:
def init_DMG():
    DMG_DIR = "./data/DMG"
    image_folder = DMG_DIR+"/images/"
    image_handler = ImageHandler("DMG", image_folder=image_folder, keep_prefix=False)

    time_stamp, pub_file, priv_file = CollectionAccessor.get_latest_dump(DMG_DIR+"/dumps")
    print(time_stamp)

    dmg_meta = dict(name="Design Museum Gent (public & private)", id_="DMG_"+time_stamp,
                creation_timestamp=time_stamp, language="nl")
    df = CollectionAccessor.get_DMG(pub_path=pub_file, #get_latest("./data/dumps", contains="public"),
                                     priv_path=priv_file, #get_latest("./data/dumps", contains="private"),
                                     rights_path=DMG_DIR+"/rights.csv",
                                     image_handler=image_handler,
                                     **dmg_meta)

    kg_searcher = GraphSearcher(df)


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=None)
    concept_search = TextEmbeddingSearcher(sem_embs, name="concept-searcher")


    sem_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/distiluse-base-multilingual-cased-v2",
                                       loadXD=32)
    sem_searcher = EmbeddingSearcher(sem_embs, name="semantic-searcher")
    
    viz_embs = EmbeddingSpaceAccessor.load(DMG_DIR+"/generated_data/vitmae", loadXD=32)
    viz_searcher = EmbeddingSearcher(viz_embs, name="visual-searcher")

    s = Search([kg_searcher, sem_searcher, viz_searcher])
    return df, s, concept_search, kg_searcher


In [ ]:
df, s, cs, graph_searcher = init_DMG()

In [ ]:
# searches = {c.attrs["id_"]: s for c, s in zip([df], [s])}
# # concept_searches = {c.attrs["id_"]: cs for c, cs in zip(collections, concept_searches)}
# collections = {c.attrs["id_"]: c for c in [df]}


## dev: Weighted edges

In [ ]:
from itertools import combinations

collection = df

pbar = tqdm(collection[collection.coll.categorical_cols.values()].fillna("").iterrows(), 
                    total=len(collection), desc='[GraphSearcher]: building graph...')
cat_obj_links = [(r.name, v) for i, r in pbar for v in GraphSearcher.iter_values(r) if v]
        
pbar = tqdm(collection[collection.coll.categorical_cols.values()].fillna("").iterrows(), 
                    total=len(collection), desc='[GraphSearcher]: building graph...')
cat_cat_links = [tuple(sorted((v1, v2)))for i, r in pbar 
                         for v1, v2 in combinations(GraphSearcher.iter_values(r), r=2)
                         if (v1 and v2) and (not v1 == v2)]

G = nx.from_edgelist(cat_obj_links+list(set(cat_cat_links)))


In [ ]:
from collections import Counter

cooc_counts = pd.Series(Counter(cat_cat_links))

max_count = cooc_counts.max()

def f(c):
    return max_count/c


weights = f(cooc_counts)

nx.set_edge_attributes(G, weights.to_dict(), name="inverse_cooccurrence_count")

In [ ]:
r = df.sample(1).iloc[0]
nx.shortest_path_length(G, source=r.name, target=None, weight="inverse_cooccurrence_count")

In [ ]:
cooc_counts.value_counts()#.sort_index()#.value_counts()

# cooc_counts.sort_values()


plt.plot(cooc_counts.value_counts().index, cooc_counts.value_counts().values, ".")

## -- end dev: Weighted edges --

---

In [ ]:
r = df.sample(1)
r

In [ ]:
scores = s(r, searcher_ids=["graph-searcher"])
ordered = s.order(df, scores)

In [ ]:
nx.shortest_path(graph_searcher.G, source=r.iloc[0].name, target="0008")

In [ ]:
ordered.loc["0008"]

In [ ]:
for o in tqdm(graph_searcher.obj_nodes):
    list(nx.all_simple_paths(graph_searcher.G, source=r.iloc[0].name, target=o, cutoff=2))

---

In [ ]:
r = df.sample(1)
GraphSearcher._build(None, r).nodes

In [ ]:
scores = s(r, searcher_ids=["graph-searcher"])

scores.value_counts().sort_index()

In [ ]:
scores[scores < 0.000040].sort_values().index

In [ ]:
GraphSearcher._build(None, df.loc[['1956']]).nodes

In [ ]:
lengths = nx.shortest_path_length(graph_searcher.G, source="onbekend", target=None)

In [ ]:
degs = graph_searcher.G.degree(graph_searcher.G.nodes)
plt.hist(list(dict(degs).values()))

In [ ]:
sorted(dict(degs).items(), key=lambda tup: tup[1], reverse=True)

In [ ]:
paths = []
for _ in tqdm(range(10000)):
    try:
        paths.append(nx.shortest_path(graph_searcher.G, source=df.sample(1).iloc[0].name, target="1987-0816_0-3"))
    except nx.NetworkXNoPath:
        continue
paths = pd.DataFrame(paths)

In [ ]:
ls = []
for o1 in tqdm(df.sample(1000).index):
    cur = nx.shortest_path_length(graph_searcher.G, source=o1, target=None)
    cur = [int(v) for k, v in sorted(cur.items())]
    ls.append(cur)

In [ ]:
df = pd.DataFrame(ls)

In [ ]:
df.astype(int)

---

In [ ]:
r = df.sample(1)

In [ ]:
r[r.coll.categorical_cols.values()]

In [ ]:
df.subcollection_name

In [ ]:
ls = []
for _ in tqdm(range(100)):
    r = df.sample(1)
    scores = s(r, searcher_ids=["graph-searcher"])
    ls.append(len(scores.value_counts()))

In [ ]:
plt.plot(scores.value_counts().sort_index().index, scores.value_counts().sort_index().values, "--")

In [ ]:
# df.loc[scores.sort_values().index]

In [ ]:
r = df.sample(1)
scores = s(r, searcher_ids=["graph-searcher"])
ordered = s.order(df, scores)
import matplotlib.pyplot as plt
plt.plot(list(ordered.sort_rank), ".")

In [ ]:
plt.plot(list(scores.loc[ordered.index]), list(df.loc[ordered.index].sort_rank), ".")

In [ ]:
ordered

In [ ]:
scores.value_counts().sort_index()

In [ ]:
ordered.iloc[:2][df.coll.categorical_cols.values()]

---

In [ ]:
GS = GraphSearcher(df.sample(50))
GS.G.nodes

In [ ]:
GS.obj_nodes

In [ ]:
sub_Gs = list(nx.connected_components(GS.G))

list(map(len, sub_Gs))

# sub_Gs[1]

In [ ]:
nx.shortest_path_length(GS.G, source="mozaïek", target=None)

In [ ]:
pos = nx.shell_layout(GS.G)

# plt.figure(figsize=(20,20))
# nx.draw_networkx(GS.G, pos=pos, with_labels=False, node_size=50, linewidths=0.3)
pos


In [ ]:
...

---

In [ ]:
gs = s.searchers[0]

In [ ]:
edges = [e for e in gs.G.edges(data=True) if e[0].startswith("1992-0004_0-2")]
edges

In [ ]:
gs.G["2010-0037_10-97"]

In [ ]:
[[1,2,3,4], [1,4]][1,2]

In [ ]:
list(gs.G.edges)[:100]

In [ ]:
o.coll.categorical_cols

In [ ]:
o = df.loc[["2010-0037_10-97",]]

In [ ]:
df.loc[gs(o).sort_values(kind="mergesort").iloc[-5:].index]

In [ ]:
import networkx as nx
pls = nx.shortest_path_length(gs.G, source=o.index[0], target=None)

In [ ]:
pls

In [ ]:
pd.DataFrame(list(pls.items()), columns=["n", "l"]).l.value_counts().sort_index()

In [ ]:
pbar = tqdm(df[df.coll.categorical_cols.values()].fillna("").iterrows(), 
                    total=len(df), desc='[GraphSearcher]: building graph...')
cat_obj_links = [(r.name, v) for i, r in pbar for v in GraphSearcher.iter_values(r)]


In [ ]:
cat_obj_links[:10]

In [ ]:
a = s(df.loc[["2010-0037_10-97", "2010-0037_10-97", "2010-0037_10-97", "2010-0037_10-97", "2008-0090"]])
b = s(df.loc[["2010-0037_10-97", "2008-0090"]])

In [ ]:
a.iloc[23912], b.iloc[12244]

In [ ]:
scores = s(o, searcher_ids=["graph-searcher"])

s.order(df, scores=scores)

In [ ]:
s.searchers[0].id

---